# An Outlier Is Not an Error

Article #31: nine seeded experiments, nine original figures, eighteen evidence CSV files. Every population, label, and shift is synthetic. A score is not a verified mistake.


## Define the target before considering robust estimators

- For mixture (1−ε)N(0,1)+εN(K,1), the **population mean** equals εK. Suppressing its shift is useful only if the intended target is the dominant component's location.
- Huber location here solves Σ clip((x−θ)/1, ±1.345)=0 with **known unit scale**. It is not an adaptive joint location-scale fit.
- A Cauchy distribution has a center and median but no defined mathematical mean.
- Student-t intervals target the *mixture mean*. Sign-test intervals target the *mixture median*. Their widths are not interchangeable.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
import numpy as np
from analysis import (Design, EXPERIMENTS, FIGURES, SEED, mixture_median, expected_shortfall_lognormal, sign_median_interval, run)
print('Seed:', SEED, '| experiments:', len(EXPERIMENTS))


In [ ]:
assert abs(mixture_median(0., 20.)) < 1e-9
q, es = expected_shortfall_lognormal(.99, 1.1)
assert es > q > 0
low, high = sign_median_interval(np.linspace(-2, 2, 100))
assert low < 0 < high
print('Lognormal q99, ES99:', round(q, 3), round(es, 3))


## Reproduce the study code in an isolated temporary directory

Quick mode runs three outer repetitions for most experiments (sixteen for CI coverage), with independent stable SeedSequence streams. It writes `_quick` files only and leaves canonical full-mode results untouched. Plots show ±1.96 times the outer-replication standard error, not sampling confidence intervals for a real population.


In [ ]:
import tempfile
with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    summaries = run(Design(root=root, quick=True))
    tables = sorted((root/'results').glob('*_quick.csv'))
    graphics = sorted((root/'figures').glob('*_quick.png'))
    assert len(tables) == 18 and len(graphics) == 9
    print('Evidence tables:', len(tables), '| figures:', len(graphics))
    for study, figure in zip(EXPERIMENTS, FIGURES):
        print('\n', study, '| summary cells:', len(summaries[study]))
        display(Image(filename=str(root/'figures'/f'{figure}_quick.png')))


## Interpretation boundaries

1. The contaminating distribution can be the *true* population, not an error mechanism. The robust estimate may target a different population functional than the mean.
2. Cook's D is computed from an OLS model with an intercept; high leverage alone neither establishes a typo nor a large coefficient change.
3. Robust covariance and Isolation Forest scores depend on training distribution, metric, and threshold. Neither establishes that an observation was recorded incorrectly.
4. The legitimate mixture minority in study 7 is labeled valid by construction. Its anomaly score carries no moral or validity judgment.
5. Study 8 deliberately changes a tail-risk estimand by deleting or capping valid outcomes. Study 9 recalibrates a 1% *review budget*, not a verified 1% error rate.

For sources, alternative estimands, numerical values, reporting protocols, and operational safeguards see `article.md`.
